# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
# TODO
#data size
df.shape

(1338, 7)

In [3]:
# TODO
#column names
df.columns
#df.columns.to_list()

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='str')

In [4]:
# data types
df.dtypes

age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object

## A.2. Missing values & Duplicate data

In [5]:
#Missing values
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [6]:
#Duplicate data
df.duplicated() #gọi hàm để trả về True False

0       False
1       False
2       False
3       False
4       False
        ...  
1333    False
1334    False
1335    False
1336    False
1337    False
Length: 1338, dtype: bool

In [7]:
df.duplicated #khác với cái trên, cái này chỉ lấy tên hàm

<bound method DataFrame.duplicated of       age     sex     bmi  children smoker     region      charges
0      19  female  27.900         0    yes  southwest  16884.92400
1      18    male  33.770         1     no  southeast   1725.55230
2      28    male  33.000         3     no  southeast   4449.46200
3      33    male  22.705         0     no  northwest  21984.47061
4      32    male  28.880         0     no  northwest   3866.85520
...   ...     ...     ...       ...    ...        ...          ...
1333   50    male  30.970         3     no  northwest  10600.54830
1334   18  female  31.920         0     no  northeast   2205.98080
1335   18  female  36.850         0     no  southeast   1629.83350
1336   21  female  25.800         0     no  southwest   2007.94500
1337   61  female  29.070         0    yes  northwest  29141.36030

[1338 rows x 7 columns]>

## A.3. Invalid values

In [8]:
# TODO
#Ktra tuổi < 19 hoặc quá cao
invalid_age = df[(df['age'] < 19) | (df['age'] > 100)]
print(invalid_age)

      age     sex     bmi  children smoker     region      charges
1      18    male  33.770         1     no  southeast   1725.55230
22     18    male  34.100         0     no  southeast   1137.01100
31     18  female  26.315         0     no  northeast   2198.18985
46     18  female  38.665         2     no  northeast   3393.35635
50     18  female  35.625         0     no  northeast   2211.13075
...   ...     ...     ...       ...    ...        ...          ...
1296   18    male  26.125         0     no  northeast   1708.92575
1315   18    male  28.310         1     no  northeast  11272.33139
1317   18    male  53.130         0     no  southeast   1163.46270
1334   18  female  31.920         0     no  northeast   2205.98080
1335   18  female  36.850         0     no  southeast   1629.83350

[69 rows x 7 columns]


In [9]:
# TODO
#Ktra BMI < 20 hoặc > 60 
invalid_bmi = df[(df['bmi'] < 20) | (df['bmi'] > 60)]
print(invalid_bmi)

      age     sex     bmi  children smoker     region      charges
28     23    male  17.385         1     no  northwest   2775.19215
98     56    male  19.950         0    yes  northeast  22412.64850
99     38    male  19.300         0    yes  southwest  15820.69900
128    32  female  17.765         2    yes  northwest  32734.18630
169    27    male  18.905         3     no  northeast   4827.90495
172    18    male  15.960         0     no  northeast   1694.79640
182    22    male  19.950         3     no  northeast   4005.42250
198    51  female  18.050         0     no  northwest   9644.25250
232    19  female  17.800         0     no  southwest   1727.78500
250    18    male  17.290         2    yes  northeast  12829.45510
266    40    male  19.800         1    yes  southeast  17179.52200
380    27  female  17.955         2    yes  northeast  15006.57945
395    46    male  19.855         0     no  northwest   7526.70645
410    19    male  17.480         0     no  northwest   1621.3

## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [10]:
# TODO
def bmi_category(bmi): #tạo 1 hàm tên bmi_category (tên hàm), hàm này nhận vào 1 gtri bmi (đầu vào)
    if bmi < 25:
        return 'Normal' #return: trả kqua ra
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'
    
df['bmi_group'] = df['bmi'].apply(bmi_category)
# df['bmi']: lấy cột BMI
# .apply(bmi_category): lấy từng gtri trong cột BMI đưa vào hàm bmi_category
# kqua sẽ lưu vào cột mới bmi_group

df['bmi_group'].value_counts() #đếm số lần xuất hiện của từng nhóm


bmi_group
Obese         707
Overweight    386
Normal        245
Name: count, dtype: int64

---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [11]:
# TODO
# Central Tendency = xu hướng trung tâm
# [mean(tbinh), media(trung vị), mode(gtri xuất hiện nhiều nhất)]

# tính gtri tbinh 
mean_charges= df['charges'].mean() 
#df['charges']: lấy cột charges
#.mean(): lấy gtri tbinh (rồi gán vào mean_charges)
print('**Mean charges:', mean_charges)

# mode_charges = df['charges'].mode()[0]

**Mean charges: 13270.422265141257


In [12]:
df.describe() #tạo bảng

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [13]:
# tính trung vị 
median_charges = df['charges'].median()
print('**Median charges:', median_charges)

**Median charges: 9382.033


In [14]:
# tính gtri xuất hiện nhiều nhất
mode_charges = df['charges'].mode()[0] #[0]: lấy gtri đầu tiên
#df['charges'].mode(): tìm giá trị xuất hiện nhiều nhất trong cột charges
print('**Mode charges:', mode_charges)

**Mode charges: 1639.5631


## Group 2 — Dispersion

In [15]:
# TODO
#Dispersion (độ phân tán): để xem gtri charges phân tán rộng hay tập trung quanh gtri tbinh
var_charges = df['charges'].var() #.var(): tính phương sai 
#(Variance lớn: dữ liệu phân tán nhiều; Variance nhỏ: dữ liệu tập trung hơn)
std_charges = df['charges'].std() #.std(): tính độ lệch chuẩn

range_charges = df['charges'].max() - df['charges'].min() #tính khoảng biến thiên

print("**Variance:", var_charges)
print("**Standard Deviation:", std_charges)
print("**Range:", range_charges)

**Variance: 146652372.15285498
**Standard Deviation: 12110.011236694001
**Range: 62648.554110000005


## Group 3 — Location and Shape

In [16]:
# TODO
# Location and Shape: xem vị trí của dữ liệu và hình dạng phân phối của dữ liệu
quantiles = df['charges'].quantile([0.25, 0.5, 0.75]) # .quantile(): dùng để tìm các phân vị

skewness = df['charges'].skew() #.skew(): đo xem phân phối dữ liệu lệch về bên nào
kurtosis = df['charges'].kurtosis() #.kurtosis(): hình dạng đuôi và mức hiện gtri cực đoan

print("**Quantiles:\n", quantiles)
print("**Skewness:", skewness) #kqua ra 1.5 thì lệch phải khá rõ
print("**Kurtosis:", kurtosis)

**Quantiles:
 0.25     4740.287150
0.50     9382.033000
0.75    16639.912515
Name: charges, dtype: float64
**Skewness: 1.5158796580240388
**Kurtosis: 1.6062986532967907


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [29]:
# TODO
# Trung bình charges theo smoker
df.groupby('smoker')['charges'].mean()

# Trung bình charges theo smoker và region
df.groupby(['smoker','region'])['charges'].mean()

smoker  region   
no      northeast     9165.531672
        northwest     8556.463715
        southeast     8032.216309
        southwest     8019.284513
yes     northeast    29673.536473
        northwest    30192.003182
        southeast    34844.996824
        southwest    32269.063494
Name: charges, dtype: float64

## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [ ]:
# TODO
# tương quan (corr) giữa BMI và charges theo nhóm smoker
# tương quan là xem khi 1 biến thay đổi thì biến kia có xu hướng thay đổi theo không
df.groupby('smoker').apply(lambda g: g['bmi'].corr(g['charges']))
# .apply(): tính cho từng nhóm 
# lambda là cách viết một hàm ngắn gọn 

smoker
no     0.084037
yes    0.806481
dtype: float64

## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [ ]:
# TODO
df.groupby('region')['charges'].mean().sort_values(ascending=False) #sắp theo lớn đến bé
#chia dữ liệu thành từng nhóm theo region, chỉ lấy cột charges

region
southeast    14735.411438
northeast    13406.384516
northwest    12417.575374
southwest    12346.937377
Name: charges, dtype: float64

## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [35]:
# TODO
df.groupby('children')['charges'].mean()
#chia dữ liệu theo children, rồi tính charges tbinh tương ứng với mõi con số

children
0    12365.975602
1    12731.171832
2    15073.563734
3    15355.318367
4    13850.656311
5     8786.035247
Name: charges, dtype: float64

## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [ ]:
# TODO
# tương quan giữa age và charges
df['age'].corr(df['charges']) #df['charges']: cột chi phí
# xem tuổi và charges có liên hệ với nhau ntn

np.float64(0.29900819333064776)

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*